# Modul 08: Reduksi Dimensi dan Analisis Faktor (PCA)
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📖 1. Reduksi Dimensi dan Analisis Faktor (PCA)

Dataset modern sering kali memuat puluhan hingga ratusan fitur prediktor (*High-Dimensional Data*). Kondisi ini menimbulkan fenomena **Kutukan Dimensi (*Curse of Dimensionality*)**, peningkatan biaya komputasi, dan ancaman multikolinearitas. **Principal Component Analysis (PCA)** adalah teknik transformasi ortogonal linier untuk mereduksi dimensi data tanpa kehilangan informasi penting:
1. **Dekomposisi Matriks Kovarians / Vektor Eigen**:
   - PCA mencari arah (*Principal Components* / PC) yang memaksimalkan varians data secara berurutan.
   - Komponen Utama Pertama (PC-1) menangkap varians terbesar, diikuti PC-2 yang tegak lurus (ortogonal 90°) terhadap PC-1, dan seterusnya.
2. **Kriteria Seleksi Komponen**:
   - **Kriteria Kaiser**: Mempertahankan komponen yang memiliki nilai *Eigenvalue* $\ge 1.0$ (varians lebih besar daripada satu variabel terstandarisasi asli).
   - **Scree Plot**: Visualisasi grafik kurva *Eigenvalue* untuk mendeteksi titik siku (*elbow point*).
   - **Cumulative Explained Variance**: Memilih jumlah komponen yang mampu mempertahankan minimal 70% - 85% total variabilitas data asli.


## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Principal Component Analysis](images/img_08_pca_reduction.png)

```
        +-------------------------------------------------------------+
        |                 PRINSIP PROYEKSI SUMBU UTAMA (PCA)          |
        +-------------------------------------------------------------+
        |  Fitur Asli (X1, X2, X3, ..., Xp)                           |
        |        |                                                    |
        |        +---> [Dekomposisi Kovarians & Vektor Eigen]         |
        |        |                                                    |
        |  Komponen Terkompresi (PC-1 [72% Var] & PC-2 [18% Var])     |
        +-------------------------------------------------------------+
```


## 🔬 3. Studi Kasus & Penjelasan Langkah Komputasi

Studi kasus mereduksi 6 fitur metrik perilaku digital pelanggan (`06_dim_reduction_customer_features.csv`) menjadi 2 komponen utama untuk visualisasi 2D dan kompresi fitur input model.

**Tahapan Komputasi:**
1. Melakukan standardisasi skala data (*StandardScaler* $Z \sim N(0, 1)$).
2. Mengekstraksi seluruh Principal Components menggunakan `sklearn.decomposition.PCA`.
3. Menghitung rasio varians, membangun Scree Plot, dan memetakan data pada ruang 2D (PC-1 vs PC-2).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_cust_feat = pd.read_csv("../datasets/06_dim_reduction_customer_features.csv")
print("Dataset multidimensi dimuat:", df_cust_feat.shape)
display(df_cust_feat.head())


## 💻 4. Eksekusi Komputasi Python & Dekomposisi Sumbu Utama


In [ ]:
# 1. Standardisasi Fitur
features = ['app_screen_time_mins', 'in_app_purchases_count', 'social_shares_count', 
            'notification_clicks', 'customer_support_tickets', 'loyalty_points_earned']
X_raw = df_cust_feat[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# 2. Ekstraksi PCA Penuh
pca_full = PCA()
pca_full.fit(X_scaled)

eigenvalues = pca_full.explained_variance_
exp_var_ratio = pca_full.explained_variance_ratio_ * 100
cum_var = np.cumsum(exp_var_ratio)

pca_summary = pd.DataFrame({
    'Principal Component': [f'PC-{i+1}' for i in range(len(features))],
    'Eigenvalue': eigenvalues,
    'Explained Variance (%)': exp_var_ratio,
    'Cumulative Variance (%)': cum_var
})
print("=== Ringkasan Dekomposisi PCA & Kriteria Kaiser ===")
display(pca_summary.round(2))


In [ ]:
# 3. Visualisasi Scree Plot dan 2D Projection Biplot
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)
df_cust_feat['PC1'] = X_pca_2d[:, 0]
df_cust_feat['PC2'] = X_pca_2d[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scree Plot
axes[0].bar(pca_summary['Principal Component'], pca_summary['Eigenvalue'], color='#1A365D', alpha=0.85)
axes[0].axhline(1.0, color='#EA580C', linestyle='--', label='Ambang Kaiser (Eigenvalue ≥ 1.0)')
axes[0].set_title('Scree Plot Pemilihan Jumlah Komponen', fontweight='bold')
axes[0].set_ylabel('Eigenvalue')
axes[0].legend()

# 2D Biplot Sebaran Data
sns.scatterplot(data=df_cust_feat, x='PC1', y='PC2', color='#2B6CB0', s=70, ax=axes[1])
axes[1].axhline(0, color='gray', linestyle=':')
axes[1].axvline(0, color='gray', linestyle=':')
axes[1].set_title(f'Proyeksi 2D PCA (Total Varians: {cum_var[1]:.1f}%)', fontweight='bold')
axes[1].set_xlabel(f'PC-1: Aktivitas Digital ({exp_var_ratio[0]:.1f}%)')
axes[1].set_ylabel(f'PC-2: Loyalitas & Tiket ({exp_var_ratio[1]:.1f}%)')

plt.tight_layout()
plt.show()


## 📝 5. Kesimpulan Analisis & Data Storytelling

### ❓ Pertanyaan Refleksi & Konsep
* **Mengapa data wajib distandarisasi (*StandardScaler*) sebelum menjalankan PCA?** Karena PCA sangat sensitif terhadap skala satuan. Fitur dengan rentang angka besar (misal poin loyalitas $0 - 10.000$) akan mendominasi arah varians secara semu dibanding fitur bernilai kecil ($0 - 10$).

### 🔍 Temuan Utama Data (Key Findings)
* Dua komponen utama pertama (**PC-1 dan PC-2**) memiliki Eigenvalue $> 1.0$ dan berhasil merangkum **90.5% total varians** dari 6 variabel asli.
* Reduksi dimensi sukses mengompresi data sebesar **66.7%** tanpa kehilangan sinyal informasi yang berarti.

### 💡 Rekomendasi & Langkah Lanjutan
* Dua fitur terkompresi (PC-1 & PC-2) dapat langsung diumpankan ke algoritma K-Means Clustering pada Modul 09.
